# Bondora P2P Loan Default Prediction: leakage-free, out-of-time pipeline

**Question:** can we predict, at the moment a loan is issued, whether it will default within 12 months, and does that beat the platform's own risk score (`ProbabilityOfDefault`)?

**How this notebook is organised:** the 12 stages required by the assignment guidelines, in order. Every number printed below comes from running the cell beneath it; nothing is typed in by hand. The headline protocol is:

| Split | Origination years | Role |
|---|---|---|
| Train | 2013 to 2018 | fit models |
| Validation | 2019 | choose model, hyper-parameters, decision threshold |
| Test | 2020 (loans with a complete 12-month window) | scored **once**, at the end |

Section 15 repeats the experiment for other window widths (6, 12, 18, 24 months) and for 2019 as the test year, using the companion scripts in `code/`.

## 1. Problem definition and use case

* **Problem:** peer-to-peer investors carry the credit risk of every loan they fund. Bondora publishes its own score, but investors want to know how much default risk can be predicted from application-time data alone, and whether an independent model ranks borrowers better.
* **Stakeholders:** retail investors (which loans to fund), the platform (scoring and pricing), borrowers (access to credit).
* **Objective:** binary classification, `default_within_12m` = 1 if the loan's `DefaultDate` falls in `[LoanDate, LoanDate + 12 months)`.
* **Inputs / outputs:** information known when the loan is issued (borrower profile, loan terms, credit scores, history before the loan) in; a default probability and a fund / skip decision out.
* **Constraints:** no information from after issuance may be used (leakage); evaluation must be chronological, because a lender only ever scores future loans.
* **Expected impact:** fewer funded loans that default, measured as the default rate of the loans a threshold would fund versus funding everything (Section 14).

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, balanced_accuracy_score, brier_score_loss,
                             confusion_matrix, classification_report, roc_curve, precision_recall_curve)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (9, 4.5); plt.rcParams["axes.grid"] = True

RANDOM_STATE = 42                 # fixed everywhere
WINDOW_MONTHS = 12                # prediction horizon (Section 15 varies it)
TEST_YEAR = 2020                  # validation = TEST_YEAR-1, train = 2013 .. TEST_YEAR-2

candidates = [os.environ.get("BONDORA_DATA"), "../data/LoanData_Bondora.csv.zip", "../data/LoanData_Bondora.csv",
              "LoanData_Bondora.csv.zip", "LoanData_Bondora.csv"]
DATA_PATH = next(p for p in candidates if p and Path(p).exists())
OUT = Path(os.environ.get("BONDORA_OUT", "../results/notebook_run")); OUT.mkdir(parents=True, exist_ok=True)
FIG = Path(os.environ.get("BONDORA_FIG", "../figures")); FIG.mkdir(parents=True, exist_ok=True)
MODEL_DIR = Path(os.environ.get("BONDORA_MODEL", "../model")); MODEL_DIR.mkdir(parents=True, exist_ok=True)
T0 = time.time()
print("data:", DATA_PATH, "| seed:", RANDOM_STATE, "| window:", WINDOW_MONTHS, "months | test year:", TEST_YEAR)

## 2. Data acquisition and understanding

* **Source:** the public Bondora loan dataset (Kaggle copy `bondora-peer-to-peer-lending-loan-data`, file `LoanData_Bondora.csv`). Bondora is the data owner. *Usage and licence terms should be checked by the team before the raw file is redistributed, which is why the raw file is not bundled with this submission.*
* **Granularity:** one row per loan.
* **Limitations to keep in mind:** a single snapshot (loans issued late have had little time to default), Bondora's own scores are model outputs of unknown provenance, and several borrower fields stopped being collected part-way through the period (shown in Section 7).

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print("shape:", df.shape)
print("exact duplicate rows:", int(df.duplicated().sum()), "| unique LoanId:", df["LoanId"].nunique())
print("column types:", df.dtypes.astype(str).value_counts().to_dict())
miss = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print("\nTop 12 columns by % missing:")
print(miss.head(12).to_string())
print("\nentirely empty columns:", [c for c in df.columns if df[c].isna().all()])

## 3. Data preparation

Dates are parsed, the five numeric fields that arrive as text are coerced to numbers, and impossible values are counted. There are no duplicate loans, so nothing is removed here. Identifier and date columns are excluded later in the leakage audit rather than deleted, so the raw frame stays intact.

In [ ]:
date_cols = ["LoanDate", "DefaultDate", "ListedOnUTC", "BiddingStartedOn", "LoanApplicationStartedDate", "ContractEndDate",
             "FirstPaymentDate", "MaturityDate_Original", "MaturityDate_Last", "LastPaymentOn", "StageActiveSince"]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

FORCE_NUMERIC = ["CreditScoreEsMicroL", "CreditScoreEsEquifaxRisk", "CreditScoreFiAsiakasTietoRiskGrade",
                 "CreditScoreEeMini", "NrOfDependants"]
before = df[FORCE_NUMERIC].notna().sum()
for c in FORCE_NUMERIC:
    df[c] = pd.to_numeric(df[c], errors="coerce")
print("values lost when coercing the 5 text columns to numeric:", (before - df[FORCE_NUMERIC].notna().sum()).to_dict())

CUTOFF = pd.to_datetime(df["ReportAsOfEOD"]).max()
print("snapshot date (ReportAsOfEOD):", CUTOFF.date())
print("LoanDate range:", df["LoanDate"].min().date(), "to", df["LoanDate"].max().date())
print("loans with Age == 0 (impossible):", int((df["Age"] == 0).sum()), "of", len(df),
      "- left as is: 0.03% of rows, tree models are insensitive, and it keeps the pipeline identical to the companion scripts")

## 4. Target definition and validation

`default_within_12m = 1` if `LoanDate <= DefaultDate < LoanDate + 12 months`. A loan is **usable** only if its whole window has ended by the snapshot date, so every usable loan was watched for the full horizon. Loans issued too recently are dropped, never labelled 0.

*Note on the eligibility rule:* the original submission also kept loans with an unfinished window if they had already defaulted, while dropping their non-defaulting peers. That inflates the default rate of the latest cohort. This notebook applies the single rule above to every split.

In [ ]:
df["window_end"] = df["LoanDate"] + pd.DateOffset(months=WINDOW_MONTHS)
valid_default_date = df["DefaultDate"].notna() & (df["DefaultDate"] >= df["LoanDate"])
default_in_window = valid_default_date & (df["DefaultDate"] < df["window_end"])
df["default_within_12m"] = default_in_window.astype(int)
df["eligible"] = df["window_end"] <= CUTOFF

print("loans:", len(df), "| usable (full window observed):", int(df["eligible"].sum()),
      "| dropped as too recent:", int((~df["eligible"]).sum()))
elig = df[df["eligible"]]
print("target among usable loans:", elig["default_within_12m"].value_counts().to_dict(),
      f"({elig['default_within_12m'].mean():.1%} default)")
print("latest usable origination date:", elig["LoanDate"].max().date())

bad1 = elig[(elig["default_within_12m"] == 1) & ~((elig["DefaultDate"] >= elig["LoanDate"]) & (elig["DefaultDate"] < elig["window_end"]))]
bad0 = elig[(elig["default_within_12m"] == 0) & elig["DefaultDate"].notna() & (elig["DefaultDate"] < elig["window_end"]) & (elig["DefaultDate"] >= elig["LoanDate"])]
print("boundary violations (default outside its window):", len(bad1), "| non-defaults with an in-window default:", len(bad0))
late = elig[(elig["default_within_12m"] == 0) & elig["DefaultDate"].notna()]
print(f"label noise: {len(late)} usable loans ({len(late) / int((elig['default_within_12m'] == 0).sum()):.1%} of the zeros) default AFTER month {WINDOW_MONTHS}")

## 5. Leakage audit and feature selection

Every predictor must be knowable when the loan is issued. The audit below excludes the target and its ingredients, everything that only exists because the loan later performed (payments, balances, recoveries, status, delinquency), servicing-schedule fields, identifiers, raw dates, and Bondora's own risk outputs (`Rating`, `ProbabilityOfDefault`, `ExpectedLoss`, `ExpectedReturn`, ...). The Bondora score is excluded because its provenance is not guaranteed to be issuance-time-only, and it is used instead as an **external baseline** to beat.

In [ ]:
LEAKY_OR_ID = {
    "default_within_12m", "window_end", "eligible", "DefaultDate", "ReportAsOfEOD", "LoanId", "LoanNumber", "UserName",
    "LastPaymentOn", "DebtOccuredOn", "DebtOccuredOnForSecondary", "CurrentDebtDaysPrimary", "CurrentDebtDaysSecondary",
    "PrincipalOverdueBySchedule", "PlannedPrincipalPostDefault", "PlannedInterestPostDefault", "EAD1", "EAD2",
    "PrincipalRecovery", "InterestRecovery", "RecoveryStage", "StageActiveSince", "ModelVersion", "Status",
    "Restructured", "ActiveLateCategory", "WorseLateCategory", "PrincipalPaymentsMade",
    "InterestAndPenaltyPaymentsMade", "PrincipalWriteOffs", "InterestAndPenaltyWriteOffs", "PrincipalBalance",
    "InterestAndPenaltyBalance", "ActiveLateLastPaymentCategory", "GracePeriodStart", "GracePeriodEnd",
    "NextPaymentDate", "NextPaymentNr", "NrOfScheduledPayments", "ReScheduledOn", "PrincipalDebtServicingCost",
    "InterestAndPenaltyDebtServicingCost", "PlannedPrincipalTillDate", "PlannedInterestTillDate",
    "ActiveScheduleFirstPaymentReached",
    "ProbabilityOfDefault", "LossGivenDefault", "ExpectedLoss", "ExpectedReturn", "Rating", "EL_V0", "Rating_V0",
    "EL_V1", "Rating_V1", "Rating_V2",
    "LoanDate", "LoanApplicationStartedDate", "ListedOnUTC", "BiddingStartedOn", "ContractEndDate", "FirstPaymentDate",
    "MaturityDate_Original", "MaturityDate_Last", "DateOfBirth", "County", "City", "ApplicationSignedHour",
    "ApplicationSignedWeekday",
}
candidate_features = [c for c in df.columns if c not in LEAKY_OR_ID]
print("columns:", df.shape[1], "| excluded:", len(LEAKY_OR_ID & set(df.columns)), "| candidate predictors:", len(candidate_features))
print(candidate_features)
pd.DataFrame({"excluded_column": sorted(LEAKY_OR_ID & set(df.columns))}).to_csv(OUT / "leakage_exclusion_audit.csv", index=False)

## 6. Train / validation / test split

The split is by **origination date**, never random: a lender can only train on the past and score the future. Validation is the year just before the test year, so it is the closest available proxy for the test population.

In [ ]:
data = df[df["eligible"]].sort_values("LoanDate").reset_index(drop=True)
yr = data["LoanDate"].dt.year
train = data[(yr >= 2013) & (yr <= TEST_YEAR - 2)].copy()
val = data[yr == TEST_YEAR - 1].copy()
test = data[yr == TEST_YEAR].copy()
split_table = pd.DataFrame([
    {"split": n, "first_loan": s["LoanDate"].min().date(), "last_loan": s["LoanDate"].max().date(), "loans": len(s),
     "defaults": int(s["default_within_12m"].sum()), "default_rate": round(s["default_within_12m"].mean(), 4)}
    for n, s in [("train", train), ("validation", val), ("test", test)]])
print(split_table.to_string(index=False))
split_table.to_csv(OUT / "split_summary.csv", index=False)
print("\nmajority-class accuracy (always 'no default'):",
      {"validation": round(1 - val["default_within_12m"].mean(), 4), "test": round(1 - test["default_within_12m"].mean(), 4)})
print("test set holds only loans with a complete window, i.e. issued", test["LoanDate"].min().date(), "to", test["LoanDate"].max().date())

## 7. Exploratory data analysis (training and validation years only)

Only years before the test year are plotted, so no test outcome can steer a modelling decision. The questions asked: how does the target move over time and by country, which fields change over time, and which borrower attributes separate defaulters?

In [ ]:
tv = pd.concat([train, val])
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
yr_rate = tv.groupby(tv["LoanDate"].dt.year)["default_within_12m"].mean()
yr_rate.plot(marker="o", ax=axes[0], color="#55a868"); axes[0].set_title("12-month default rate by origination year")
axes[0].set_xlabel("year"); axes[0].set_ylabel("default rate")
for k in ["EE", "ES", "FI"]:
    s = tv[(tv["Country"] == k) & (tv["LoanDate"].dt.year >= 2014)]
    s.groupby(s["LoanDate"].dt.year)["default_within_12m"].mean().plot(marker="o", ax=axes[1], label=k)
axes[1].set_title("by country (2014 onwards)"); axes[1].legend(); axes[1].set_xlabel("year")
train["default_within_12m"].value_counts().sort_index().plot(kind="bar", ax=axes[2], color=["#4c72b0", "#c44e52"])
axes[2].set_xticklabels(["no default", "default"], rotation=0); axes[2].set_title("training class balance")
plt.tight_layout(); plt.savefig(FIG / "eda_target_by_year_country.png", dpi=120); plt.show()
print(tv.groupby("Country")["default_within_12m"].agg(["size", "mean"]).round(3).sort_values("size", ascending=False).head(5).to_string())

In [ ]:
# Which fields stop being collected? share of loans carrying real information, by origination year
fields = {"UseOfLoan": -1, "MaritalStatus": -1, "EmploymentStatus": -1, "OccupationArea": -1,
          "IncomeFromPrincipalEmployer": 0, "DebtToIncome": 0, "FreeCash": 0}      # value that means 'not collected'
yrs = df["LoanDate"].dt.year
pop = pd.DataFrame({f: ((df[f].notna()) & (df[f] != nc)).groupby(yrs).mean() for f, nc in fields.items()})
pop = pop.loc[2013:TEST_YEAR - 1].round(2)
print(pop.to_string())
ax = pop.plot(figsize=(10, 4.2), marker="o"); ax.set_ylabel("share of loans with the field filled")
ax.set_title("Borrower fields that stop being collected (the cut-over is June 2017)"); ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(FIG / "eda_field_availability_by_year.png", dpi=120); plt.show()
m = df[(df["LoanDate"] >= "2017-04-01") & (df["LoanDate"] < "2017-08-01")]
print("\nUseOfLoan filled, by month:", ((m["UseOfLoan"] != -1).groupby(m["LoanDate"].dt.to_period("M")).mean().round(2)).to_dict())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, c in zip(axes.ravel(), ["Amount", "Interest", "IncomeTotal", "Age", "LoanDuration", "MonthlyPayment",
                                "LiabilitiesTotal", "AmountOfPreviousLoansBeforeLoan"]):
    s = train[c].dropna(); s = s.clip(upper=s.quantile(0.99))
    ax.hist(s, bins=40, color="#4c72b0"); ax.set_title(c + " (clipped at p99)", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "eda_distributions.png", dpi=110); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
t = train.groupby("Education")["default_within_12m"].agg(["mean", "count"]); t = t[t["count"] >= 200]
axes[0].bar(t.index.astype(str), t["mean"], color="#c44e52"); axes[0].set_title("default rate by Education (>=200 loans)")
t = train.groupby("NewCreditCustomer")["default_within_12m"].agg(["mean", "count"])
axes[1].bar(t.index.astype(str), t["mean"], color="#8172b2"); axes[1].set_title("default rate: new vs returning customer")
q = pd.qcut(train["Interest"], 10, duplicates="drop")
axes[2].plot(range(10), train.groupby(q)["default_within_12m"].mean().values, marker="o"); axes[2].set_title("default rate by Interest decile")
plt.tight_layout(); plt.savefig(FIG / "eda_default_rate_drivers.png", dpi=110); plt.show()

cols = ["Amount", "Interest", "IncomeTotal", "Age", "LoanDuration", "MonthlyPayment", "LiabilitiesTotal",
        "NoOfPreviousLoansBeforeLoan", "AmountOfPreviousLoansBeforeLoan", "default_within_12m"]
cm = train[cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6.5)); im = ax.imshow(cm, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=90, fontsize=7); ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=7)
fig.colorbar(im); ax.set_title("Spearman correlation (training)"); plt.tight_layout(); plt.savefig(FIG / "eda_correlation.png", dpi=110); plt.show()
print("correlation with the target:", cm["default_within_12m"].drop("default_within_12m").round(3).sort_values().to_dict())

**EDA observations that shape the modelling**

1. The default rate moves a lot over time and between countries, so the evaluation must be out-of-time and per-country results should be reported next to the pooled number.
2. A group of borrower fields (loan purpose, marital and employment status, debt-to-income, free cash, occupation, most income components) is filled until May 2017 and empty or constant afterwards, in every country at once. For loans issued from 2018 on, these columns carry no information; models cannot rely on them at scoring time. They stay in the feature list because the original approach keeps them; their effect on the results was not isolated in this notebook, and a tree model simply sees constants for recent loans.
3. No single numeric feature is strongly correlated with default; risk is multi-factorial, which favours gradient-boosted trees over a linear model.
4. Class balance is moderate (roughly 30 to 33% defaults in training), so accuracy alone is a weak yardstick; PR-AUC, recall and lift are reported too.

## 8. Data preprocessing and feature engineering

* **Numeric:** median imputation (fitted on the training years only), no scaling (tree models are scale-free).
* **Categorical** (`Country`, `EmploymentDurationCurrentEmployer`, `WorkExperience`): the literal category `missing` for NaN, then one-hot encoding for LightGBM / XGBoost / logistic regression; CatBoost uses its native categorical handling.
* **Outliers:** left unchanged. Income and liabilities are heavy-tailed, but tree models split on ranks, so extreme values do not distort them; winsorising was not tested here.
* **Engineered features** (issuance-time only, divisions guarded against zero): `dti` = MonthlyPayment / (IncomeTotal / 12), `amt_ratio` = Amount / AppliedAmount, `inc_month`, `loan_inc` = Amount / IncomeTotal, `freecash_neg`.

In [ ]:
numeric_features = [c for c in candidate_features if pd.api.types.is_numeric_dtype(train[c])]
categorical_features = [c for c in candidate_features if c not in numeric_features]


def make_matrix(d):
    X = pd.DataFrame(index=d.index)
    for c in numeric_features:
        X[c] = pd.to_numeric(d[c], errors="coerce")
    for c in categorical_features:
        X[c] = d[c].where(d[c].notna(), "missing").astype(str)
    X["dti"] = (X["MonthlyPayment"] / (X["IncomeTotal"] / 12.0)).replace([np.inf, -np.inf], np.nan)
    X["amt_ratio"] = (X["Amount"] / X["AppliedAmount"]).replace([np.inf, -np.inf], np.nan)
    X["inc_month"] = X["IncomeTotal"] / 12.0
    X["loan_inc"] = (X["Amount"] / X["IncomeTotal"]).replace([np.inf, -np.inf], np.nan)
    X["freecash_neg"] = (X["FreeCash"] < 0).astype(int)
    return X


Xtr, Xva, Xte = make_matrix(train), make_matrix(val), make_matrix(test)
ytr, yva, yte = (d["default_within_12m"].values for d in (train, val, test))
num_cols = [c for c in Xtr.columns if pd.api.types.is_numeric_dtype(Xtr[c])]
cat_cols = [c for c in Xtr.columns if c not in num_cols]


def make_preprocessor():
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                         ("ohe", OneHotEncoder(handle_unknown="ignore", max_categories=200, sparse_output=False))])
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])


print("records before / after preparation:", len(df), "->", len(data), "usable loans")
print("features: raw candidates", len(candidate_features), "-> model inputs", Xtr.shape[1],
      f"({len(num_cols)} numeric incl. 5 engineered, {len(cat_cols)} categorical: {cat_cols})")
print("matrices:", Xtr.shape, Xva.shape, Xte.shape)
print("missing values in model inputs (train):", int(Xtr.isna().sum().sum()), "-> handled by the imputer, fitted on train only")

## 9. Data balancing

The training set is moderately imbalanced. Validation and test are **never** resampled. The check below asks whether class weighting changes the *ranking* quality (ROC-AUC) of a gradient-boosted model, or only moves the operating point.

In [ ]:
print("training class balance:", pd.Series(ytr).value_counts(normalize=True).round(4).to_dict())
print("negatives per positive:", round((ytr == 0).sum() / (ytr == 1).sum(), 2))
rows = []
for label, w in [("no weighting", None), ("class_weight='balanced'", "balanced")]:
    pipe = Pipeline([("pre", make_preprocessor()),
                     ("m", LGBMClassifier(n_estimators=600, max_depth=6, learning_rate=0.03, num_leaves=31, class_weight=w,
                                          n_jobs=-1, random_state=RANDOM_STATE, verbose=-1))]).fit(Xtr, ytr)
    p = pipe.predict_proba(Xva)[:, 1]
    rows.append({"scheme": label, "val_roc_auc": roc_auc_score(yva, p), "val_pr_auc": average_precision_score(yva, p),
                 "accuracy@0.5": accuracy_score(yva, p >= .5), "recall@0.5": recall_score(yva, p >= .5),
                 "mean_predicted_risk": p.mean(), "actual_rate": yva.mean(), "brier": brier_score_loss(yva, p)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Decision:** weighting changes the operating point (recall up, accuracy down) and inflates predicted probabilities, but not the ranking quality. Where probabilities matter (calibration), no weighting is preferable; where a fixed 0.5 cut-off is used, weighting is a convenience. The models below follow the original configuration, and the decision threshold is chosen on validation either way.

## 10. Baseline models

Three reference points before any boosted model: the **majority class** (what accuracy costs nothing), a **logistic regression** (a linear, interpretable baseline), and **Bondora's own `ProbabilityOfDefault`** on the same validation loans (the external benchmark the project must beat).

In [ ]:
def thr_metrics(y, p, t):
    pred = (p >= t).astype(int)
    return {"accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred), "f1": f1_score(y, pred), "balanced_acc": balanced_accuracy_score(y, pred)}


dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
logreg = Pipeline([("pre", make_preprocessor()),
                   ("m", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5, random_state=RANDOM_STATE))]).fit(Xtr, ytr)
p_lr = logreg.predict_proba(Xva)[:, 1]
bond_val = val["ProbabilityOfDefault"].values
ok = ~np.isnan(bond_val)
base = pd.DataFrame([
    {"model": "majority class", **thr_metrics(yva, np.zeros(len(yva)), .5), "roc_auc": 0.5, "pr_auc": yva.mean()},
    {"model": "logistic regression", **thr_metrics(yva, p_lr, .5), "roc_auc": roc_auc_score(yva, p_lr), "pr_auc": average_precision_score(yva, p_lr)},
    {"model": "Bondora ProbabilityOfDefault", **thr_metrics(yva[ok], bond_val[ok], .5), "roc_auc": roc_auc_score(yva[ok], bond_val[ok]),
     "pr_auc": average_precision_score(yva[ok], bond_val[ok])}])
print("BASELINES on validation", TEST_YEAR - 1, "(threshold 0.5)")
print(base.round(4).to_string(index=False))
print("\nBondora score available for", f"{ok.mean():.1%}", "of validation loans")

## 11. Modelling: why these algorithms

* **LightGBM, XGBoost, CatBoost** (gradient-boosted trees): tabular data with mixed types, missing values, non-linear effects and interactions, and no need for scaling. They are the standard strong choice for credit-risk tabular problems.
* **CatBoost** additionally handles categorical columns natively, avoiding the sparsity of one-hot encoding.
* **Logistic regression** is kept as the transparent baseline; if boosting could not beat it, the extra complexity would not be justified.
* Hyper-parameters follow the original submission (shallow-to-medium trees, small learning rate, strong L2 regularisation, row and column subsampling). All models are fitted on the training years only; validation is used for model selection, never for fitting.

In [ ]:
LGBM_KW = dict(n_estimators=1500, max_depth=8, learning_rate=0.008, num_leaves=63, reg_alpha=0.5, reg_lambda=15,
               min_child_samples=40, subsample=0.8, colsample_bytree=0.7, class_weight="balanced", n_jobs=-1,
               random_state=RANDOM_STATE, verbose=-1)
XGB_KW = dict(n_estimators=1500, max_depth=6, learning_rate=0.008, subsample=0.8, colsample_bytree=0.7, reg_lambda=15,
              reg_alpha=0.5, min_child_weight=10, scale_pos_weight=1.5, n_jobs=-1, random_state=RANDOM_STATE, verbosity=0)
CB_KW = dict(iterations=1000, depth=6, learning_rate=0.03, l2_leaf_reg=8, random_seed=RANDOM_STATE, verbose=False,
             allow_writing_files=False)
cat_idx = [Xtr.columns.get_loc(c) for c in cat_cols]

lgbm = Pipeline([("pre", make_preprocessor()), ("m", LGBMClassifier(**LGBM_KW))]).fit(Xtr, ytr)
print(f"[{time.time() - T0:5.0f}s] LightGBM fitted")
xgb = Pipeline([("pre", make_preprocessor()), ("m", XGBClassifier(**XGB_KW))]).fit(Xtr, ytr)
print(f"[{time.time() - T0:5.0f}s] XGBoost fitted")
catboost = CatBoostClassifier(cat_features=cat_idx, **CB_KW).fit(Xtr, ytr)
print(f"[{time.time() - T0:5.0f}s] CatBoost fitted")

prob_val = {"LogisticRegression": p_lr, "LightGBM": lgbm.predict_proba(Xva)[:, 1],
            "XGBoost": xgb.predict_proba(Xva)[:, 1], "CatBoost": catboost.predict_proba(Xva)[:, 1]}
prob_tr = {"LightGBM": lgbm.predict_proba(Xtr)[:, 1], "XGBoost": xgb.predict_proba(Xtr)[:, 1], "CatBoost": catboost.predict_proba(Xtr)[:, 1]}
print("\ntrain vs validation ROC-AUC (a large gap means overfitting):")
print(pd.DataFrame({"train": {k: roc_auc_score(ytr, v) for k, v in prob_tr.items()},
                    "validation": {k: roc_auc_score(yva, prob_val[k]) for k in prob_tr}}).round(4).to_string())

## 12. Hyper-parameter tuning

A small, reproducible grid is evaluated on the **validation** year for the two families where depth and regularisation matter most. The test year is not involved. Tuning is deliberately modest (a regularised, sensible configuration rather than an exhaustive search), and the table also shows how little the choice matters.

In [ ]:
tuning = []
for params in [dict(n_estimators=800, max_depth=4, learning_rate=0.02, num_leaves=20, reg_lambda=4),
               dict(n_estimators=1500, max_depth=8, learning_rate=0.008, num_leaves=63, reg_lambda=15),
               dict(n_estimators=2000, max_depth=10, learning_rate=0.006, num_leaves=127, reg_lambda=25)]:
    kw = {**LGBM_KW, **params}
    pipe = Pipeline([("pre", make_preprocessor()), ("m", LGBMClassifier(**kw))]).fit(Xtr, ytr)
    p = pipe.predict_proba(Xva)[:, 1]
    tuning.append({"model": "LightGBM", "n_estimators": params["n_estimators"], "max_depth": params["max_depth"],
                   "learning_rate": params["learning_rate"], "val_roc_auc": roc_auc_score(yva, p)})
    print(f"[{time.time() - T0:5.0f}s] tuned LightGBM depth {params['max_depth']}")
for depth, lr, iters in [(6, 0.03, 1000), (8, 0.02, 1200)]:
    cb = CatBoostClassifier(iterations=iters, depth=depth, learning_rate=lr, l2_leaf_reg=8, random_seed=RANDOM_STATE,
                            verbose=False, allow_writing_files=False, cat_features=cat_idx).fit(Xtr, ytr)
    p = cb.predict_proba(Xva)[:, 1]
    tuning.append({"model": "CatBoost", "n_estimators": iters, "max_depth": depth, "learning_rate": lr, "val_roc_auc": roc_auc_score(yva, p)})
    print(f"[{time.time() - T0:5.0f}s] tuned CatBoost depth {depth}")
tuning_df = pd.DataFrame(tuning).sort_values("val_roc_auc", ascending=False)
print("\nTuning grid (validation ROC-AUC):")
print(tuning_df.round(4).to_string(index=False))
tuning_df.to_csv(OUT / "tuning_grid_validation.csv", index=False)

## 13. Validation: model comparison and selection

All candidates are scored on the same validation year. The decision threshold for each model is the one that maximises **validation accuracy**; precision, recall, F1 and balanced accuracy are shown beside it so the trade-off is visible. The final model is the one with the highest validation **ROC-AUC** (threshold-free ranking quality), ties broken by accuracy.

In [ ]:
def best_acc_threshold(prob, y):
    best = (-1, 0.5)
    for t in np.arange(0.05, 0.95, 0.005):
        a = accuracy_score(y, (prob >= t).astype(int))
        if a > best[0]:
            best = (a, t)
    return best[1]


rows = [{"model": "Dummy (majority)", "threshold": np.nan, **thr_metrics(yva, np.zeros(len(yva)), .5), "roc_auc": 0.5, "pr_auc": yva.mean()}]
for name, p in prob_val.items():
    t = best_acc_threshold(p, yva)
    rows.append({"model": name, "threshold": round(t, 3), **thr_metrics(yva, p, t), "roc_auc": roc_auc_score(yva, p),
                 "pr_auc": average_precision_score(yva, p)})
comparison_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
print(f"VALIDATION ({TEST_YEAR - 1}) model comparison")
print(comparison_df.round(4).to_string(index=False))
comparison_df.to_csv(OUT / "validation_model_comparison.csv", index=False)

pool = comparison_df[comparison_df["model"] != "Dummy (majority)"]
best_name = pool.sort_values(["roc_auc", "accuracy"], ascending=False).iloc[0]["model"]
sel_threshold = float(comparison_df.loc[comparison_df["model"] == best_name, "threshold"].iloc[0])
print("\nselected by validation ROC-AUC:", best_name, "| validation-tuned threshold:", sel_threshold)
gap = pool["roc_auc"].sort_values(ascending=False).values
print(f"margin between the best and second-best validation ROC-AUC: {gap[0] - gap[1]:.4f} (small margins mean the pick is close to a tie)")

## 14. Final model and the single test evaluation

After the model family, hyper-parameters and threshold are fixed on validation, the chosen model is **refitted on all pre-test data** (train plus validation) with exactly the validated hyper-parameters, because the test population is closer to the validation year than to 2013. The test year is scored once. Reported: accuracy, precision, recall, F1, balanced accuracy, ROC-AUC, PR-AUC with lift, Brier score, the confusion matrix, per-country results, and the same metrics for Bondora's own score on the same loans.

In [ ]:
X_final = pd.concat([Xtr, Xva], ignore_index=True)
y_final = np.concatenate([ytr, yva])
if best_name == "CatBoost":
    final_model = CatBoostClassifier(cat_features=[X_final.columns.get_loc(c) for c in cat_cols], **CB_KW)
elif best_name == "LightGBM":
    final_model = Pipeline([("pre", make_preprocessor()), ("m", LGBMClassifier(**LGBM_KW))])
elif best_name == "XGBoost":
    final_model = Pipeline([("pre", make_preprocessor()), ("m", XGBClassifier(**XGB_KW))])
else:
    final_model = Pipeline([("pre", make_preprocessor()), ("m", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5, random_state=RANDOM_STATE))])
final_model.fit(X_final, y_final)
print(f"[{time.time() - T0:5.0f}s] final {best_name} refitted on {len(X_final)} loans ({train['LoanDate'].min().year}-{val['LoanDate'].max().year})")

test_prob = final_model.predict_proba(Xte)[:, 1]            # the test set is scored here, once
m_thr = thr_metrics(yte, test_prob, sel_threshold)
m_half = thr_metrics(yte, test_prob, 0.5)
extra = {"roc_auc": roc_auc_score(yte, test_prob), "pr_auc": average_precision_score(yte, test_prob),
         "lift": average_precision_score(yte, test_prob) / yte.mean(), "brier": brier_score_loss(yte, test_prob)}
majority_acc = 1 - yte.mean()
print(f"\nFINAL TEST {TEST_YEAR} | model={best_name} | threshold={sel_threshold:.3f} | n={len(yte)} | default rate={yte.mean():.4f}")
print(pd.DataFrame({"at validation threshold": {**m_thr, **extra}, "at 0.5": {**m_half, **extra}}).round(4).to_string())
print(f"majority-class accuracy on test: {majority_acc:.4f}  -> accuracy edge at the validation threshold: {(m_thr['accuracy'] - majority_acc) * 100:+.2f} points")
cm = confusion_matrix(yte, (test_prob >= sel_threshold).astype(int))
print("\nconfusion matrix at the validation threshold (rows = true, cols = predicted):\n", cm)
print(classification_report(yte, (test_prob >= sel_threshold).astype(int), target_names=["no default", "default"], digits=3))

pd.DataFrame([{"model": best_name, "threshold": sel_threshold, "test_year": TEST_YEAR, "window_months": WINDOW_MONTHS, **m_thr, **extra,
               "majority_baseline_accuracy": majority_acc}]).to_csv(OUT / "final_test_results.csv", index=False)
pd.DataFrame({"y_true": yte, "prob_default": test_prob, "pred_default": (test_prob >= sel_threshold).astype(int),
              "country": test["Country"].values}).to_csv(OUT / "final_test_predictions.csv", index=False)
joblib.dump(final_model, MODEL_DIR / "final_model.pkl")
json.dump({"selected_model": best_name, "threshold": sel_threshold, "window_months": WINDOW_MONTHS, "test_year": TEST_YEAR,
           "train_years": f"2013-{TEST_YEAR - 1}", "numeric_features": num_cols, "categorical_features": cat_cols},
          open(MODEL_DIR / "metadata.json", "w"), indent=2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.6))
im = axes[0].imshow(cm, cmap="Blues"); axes[0].set_title(f"Confusion matrix (test {TEST_YEAR})")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["pred 0", "pred 1"]); axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["true 0", "true 1"]); axes[0].grid(False)

okb = ~np.isnan(test["ProbabilityOfDefault"].values)
fpr, tpr, _ = roc_curve(yte, test_prob); axes[1].plot(fpr, tpr, label=f"model AUC {extra['roc_auc']:.3f}")
bp = test["ProbabilityOfDefault"].values[okb]; fpr2, tpr2, _ = roc_curve(yte[okb], bp)
axes[1].plot(fpr2, tpr2, label=f"Bondora PD AUC {roc_auc_score(yte[okb], bp):.3f}"); axes[1].plot([0, 1], [0, 1], "k--", lw=.8)
axes[1].set_title("ROC curve"); axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].legend()

pr, rc, _ = precision_recall_curve(yte, test_prob); axes[2].plot(rc, pr, label=f"model PR-AUC {extra['pr_auc']:.3f}")
pr2, rc2, _ = precision_recall_curve(yte[okb], bp); axes[2].plot(rc2, pr2, label=f"Bondora PD PR-AUC {average_precision_score(yte[okb], bp):.3f}")
axes[2].axhline(yte.mean(), color="k", ls="--", lw=.8, label=f"base rate {yte.mean():.3f}")
axes[2].set_title("Precision-recall curve"); axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].legend()

for nm, p_, y_ in [("model", test_prob, yte), ("Bondora PD", bp, yte[okb])]:
    b = pd.qcut(pd.Series(p_).rank(method="first"), 10, labels=False)
    t = pd.DataFrame({"p": p_, "y": y_, "b": b.values}).groupby("b").mean()
    axes[3].plot(t["p"], t["y"], marker="o", label=nm)
axes[3].plot([0, 1], [0, 1], "k--", lw=.8); axes[3].set_title("Calibration by risk decile"); axes[3].set_xlabel("mean predicted"); axes[3].set_ylabel("observed default rate"); axes[3].legend()
plt.tight_layout(); plt.savefig(FIG / "final_test_diagnostics.png", dpi=120); plt.show()

In [ ]:
# Per-country results next to Bondora's own score, on the same test loans
rows = []
for k in ["ALL", "EE", "ES", "FI"]:
    msk = np.ones(len(test), bool) if k == "ALL" else (test["Country"].values == k)
    if msk.sum() < 300 or not (0 < yte[msk].mean() < 1):
        continue
    mk = msk & okb
    rows.append({"group": k, "loans": int(msk.sum()), "default_rate": yte[msk].mean(),
                 "model_roc_auc": roc_auc_score(yte[msk], test_prob[msk]), "bondora_roc_auc": roc_auc_score(yte[mk], test["ProbabilityOfDefault"].values[mk]),
                 "model_pr_auc": average_precision_score(yte[msk], test_prob[msk]),
                 "bondora_pr_auc": average_precision_score(yte[mk], test["ProbabilityOfDefault"].values[mk]),
                 "model_lift": average_precision_score(yte[msk], test_prob[msk]) / yte[msk].mean(), "mean_predicted": test_prob[msk].mean()})
by_country = pd.DataFrame(rows)
print(by_country.round(4).to_string(index=False))
by_country.to_csv(OUT / "final_test_by_country.csv", index=False)

# Use case: what would funding by this model have changed? (test loans, ranked by predicted risk)
d = pd.DataFrame({"p": test_prob, "y": yte, "bond": test["ProbabilityOfDefault"].values})
d["decile"] = pd.qcut(d["p"].rank(method="first"), 10, labels=False)
print("\nDefault rate by predicted-risk decile (0 = safest):")
print(d.groupby("decile").agg(loans=("y", "size"), actual_default_rate=("y", "mean"), mean_predicted=("p", "mean")).round(3).to_string())
rows = []
for share in [0.1, 0.2, 0.3, 0.5]:
    for nm, col in [("model", "p"), ("Bondora PD", "bond")]:
        s = d.dropna(subset=[col]); kept = s[s[col] <= s[col].quantile(1 - share)]
        rows.append({"skip riskiest": f"{int(share * 100)}%", "ranked by": nm, "default rate of funded loans": round(kept["y"].mean(), 3),
                     "share of all defaults avoided": round(1 - kept["y"].sum() / s["y"].sum(), 3), "default rate if fund everything": round(s["y"].mean(), 3)})
sim = pd.DataFrame(rows)
print("\nFunding simulation:"); print(sim.to_string(index=False))
sim.to_csv(OUT / "funding_simulation.csv", index=False)

In [ ]:
if best_name == "CatBoost":
    imp = pd.Series(final_model.get_feature_importance(), index=X_final.columns)
else:
    imp = pd.Series(final_model.named_steps["m"].feature_importances_, index=final_model.named_steps["pre"].get_feature_names_out())
imp = imp.sort_values(ascending=False)
imp.to_csv(OUT / "feature_importance.csv", header=["importance"])
top = imp.head(15)[::-1]
plt.figure(figsize=(9, 5.5)); plt.barh(top.index.astype(str), top.values, color="#4c72b0")
plt.title(f"Top 15 native feature importances ({best_name})"); plt.tight_layout(); plt.savefig(FIG / "feature_importance.png", dpi=120); plt.show()
print(imp.head(15).round(2).to_string())
print("\nAll top drivers are application-time variables (loan terms, borrower income / liabilities, history before the loan); none is a post-issuance field.")

## 15. Robustness: other window widths and the previous year as test

The same protocol was run by the scripts in `code/` for window widths of 6, 12, 18 and 24 months and for **two test years**: 2020 (the headline protocol above) and 2019 (train 2013 to 2017, validate 2018, refit through 2018, test 2019). A window is only usable when it has fully elapsed before the snapshot, so some cells are partial years or not possible. **Metrics are not comparable across window widths** because the base rate and the test loans change; each row must be read against its own Bondora and majority-class references.

In [ ]:
res_candidates = [Path(os.environ.get("BONDORA_RESULTS", "../results")), Path("../results"), Path("zip_window_compare"), Path(".")]
frames = []
for T in (2019, 2020):
    f = next((r / f"window_compare_test{T}.csv" for r in res_candidates if (r / f"window_compare_test{T}.csv").exists()), None)
    if f is not None:
        frames.append(pd.read_csv(f))
if not frames:
    print("window_compare_test*.csv not found - run code/zip_approach_window_compare.py first")
else:
    W = pd.concat(frames)
    W["roc_gain_vs_bondora"] = W["roc_auc"] - W["bondora_pd_roc_auc"]
    W["accuracy_edge_pts"] = (W["acc_at_val_thr"] - W["test_majority_acc"]) * 100
    cols = ["test_year", "window_months", "status", "n_test", "rate_test", "selected", "roc_auc", "bondora_pd_roc_auc",
            "roc_gain_vs_bondora", "pr_auc", "lift", "acc_at_val_thr", "test_majority_acc", "accuracy_edge_pts", "recall_at_val_thr"]
    print(W[cols].round(3).to_string(index=False))
    ok_ = W[W["roc_auc"].notna()]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
    for ax, T in zip(axes, (2019, 2020)):
        s = ok_[ok_["test_year"] == T]; x = np.arange(len(s))
        ax.bar(x - .2, s["roc_auc"], .4, label="model", color="#4c72b0"); ax.bar(x + .2, s["bondora_pd_roc_auc"], .4, label="Bondora PD", color="#c44e52")
        ax.set_xticks(x); ax.set_xticklabels([f"{int(w)} m" for w in s["window_months"]]); ax.set_ylim(.5, .85)
        ax.set_title(f"ROC-AUC by window width, test {T}"); ax.legend()
    plt.tight_layout(); plt.savefig(FIG / "window_sweep_roc_auc.png", dpi=120); plt.show()

## 16. Impact, limitations and future work

**Who benefits and how it would be used.** An investor (or the platform) ranks incoming loans by predicted default risk and declines the riskiest slice. The funding simulation in Section 14 reports what that would have done to the default rate of the funded loans on the test year, next to what Bondora's own score would have done. The model improves *ranking* over Bondora's score; it is not a calibrated probability, so thresholds should be set from a recent validation period rather than read off the raw output.

**Limitations (stated plainly)**

1. *Absolute accuracy is modest.* In every setting the accuracy edge over "always predict no default" is small at a 12-month window, because the base rate is already 70 to 80% non-default. Ranking metrics (ROC-AUC, PR-AUC, lift) are the right yardstick and are reported next to accuracy.
2. *Label noise.* A 12-month window labels loans that default later as non-defaults (roughly a third of the zeros, measured in Section 4); longer windows reduce this but remove recent loans from the test set.
3. *Temporal drift.* Default rates and country mix change by year; probabilities from a class-weighted model are inflated, so the model should be re-validated on the latest year before use.
4. *Discontinued fields.* Several borrower fields were not collected from June 2017; the model cannot learn from them for recent loans.
5. *Single snapshot.* Only the status at the snapshot date is known, so lifetime monitoring (re-scoring a loan as it ages) is out of scope.
6. *Model selection margin.* The three boosted models differ by a few thousandths of ROC-AUC on validation; the selected model could change between years without any real difference in quality.

**Future work.** Survival or discrete-time models that use all loans including unfinished ones; per-country calibration on the latest year; expected-loss targets that weigh the size of the loss, not just whether a default occurred; and monitoring for drift once deployed.

**Academic integrity.** All figures come from the code in this notebook. Code was developed with AI assistance and reviewed and run by the team; the team can explain every step. No result was edited by hand, and the test year was scored once.